In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Singularity Machine Learning - Classification: Funkcja Qiskit od Multiverse Computing
> **Note:** * Funkcje Qiskit to funkcja eksperymentalna dostępna wyłącznie dla użytkowników planów IBM Quantum&reg; Premium Plan, Flex Plan oraz On-Prem (za pośrednictwem interfejsu API IBM Quantum Platform). Mają status wersji zapoznawczej i mogą ulec zmianie.

## Przegląd
Funkcja „Singularity Machine Learning - Classification" pozwala rozwiązywać rzeczywiste problemy uczenia maszynowego na sprzęcie kwantowym bez konieczności posiadania wiedzy z zakresu informatyki kwantowej. Ta funkcja aplikacyjna, oparta na metodach zespołowych, jest hybrydowym klasyfikatorem. Wykorzystuje klasyczne metody, takie jak boosting, bagging i stacking, do wstępnego trenowania zespołu. Następnie stosuje algorytmy kwantowe — wariacyjny solver własny (VQE) oraz kwantowy algorytm przybliżonej optymalizacji (QAOA) — w celu zwiększenia różnorodności wytrenowanego zespołu, jego zdolności do generalizacji oraz ogólnej złożoności.

W przeciwieństwie do innych rozwiązań z zakresu kwantowego uczenia maszynowego, ta funkcja potrafi obsługiwać wielkoskalowe zbiory danych zawierające miliony przykładów i cech, nie będąc ograniczona liczbą Qubitów w docelowym QPU. Liczba Qubitów określa jedynie rozmiar zespołu, który można wytrenować. Funkcja jest również bardzo elastyczna i może być stosowana do rozwiązywania problemów klasyfikacyjnych w szerokim zakresie dziedzin, w tym finansów, opieki zdrowotnej i cyberbezpieczeństwa.
Konsekwentnie osiąga wysoką dokładność na klasycznie trudnych problemach obejmujących wielowymiarowe, zaszumione i niezrównoważone zbiory danych.
![Jak to działa](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
Przeznaczona jest dla:
1. Inżynierów i naukowców zajmujących się danymi w firmach, którzy chcą wzbogacić swoją ofertę technologiczną, integrując kwantowe uczenie maszynowe ze swoimi produktami i usługami,
2. Badaczy w laboratoriach badań kwantowych eksplorujących zastosowania kwantowego uczenia maszynowego i pragnących wykorzystać obliczenia kwantowe do zadań klasyfikacyjnych, oraz
3. Studentów i nauczycieli w instytucjach edukacyjnych, na kursach takich jak uczenie maszynowe, którzy chcą zademonstrować przewagi obliczeń kwantowych.

Poniższy przykład prezentuje różne funkcjonalności, w tym `create`, `list`, `fit` i `predict`, oraz demonstruje ich użycie na syntetycznym problemie złożonym z dwóch przeplatających się półokręgów — notorrycznie trudnym ze względu na nieliniową granicę decyzyjną.


## Opis funkcji
Ta funkcja Qiskit umożliwia użytkownikom rozwiązywanie binarnych problemów klasyfikacyjnych za pomocą kwantowo ulepszonego klasyfikatora zespołowego Singularity. W tle wykorzystuje podejście hybrydowe: klasycznie trenuje zespół klasyfikatorów na oznaczonym zbiorze danych, a następnie optymalizuje go pod kątem maksymalnej różnorodności i generalizacji przy użyciu Kwantowego Algorytmu Przybliżonej Optymalizacji (QAOA) na QPU IBM&reg;. Dzięki przyjaznemu interfejsowi użytkownicy mogą skonfigurować klasyfikator zgodnie ze swoimi wymaganiami, wytrenować go na wybranym zbiorze danych i użyć do przewidywania na wcześniej niewidzianym zbiorze danych.

Aby rozwiązać ogólny problem klasyfikacyjny:
1. Przygotuj wstępnie zbiór danych i podziel go na zestawy treningowy i testowy. Opcjonalnie możesz dalej podzielić zestaw treningowy na zestawy treningowy i walidacyjny. Można to osiągnąć za pomocą [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. Jeśli zestaw treningowy jest niezrównoważony, możesz go próbkować ponownie, aby wyrównać klasy, używając [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. Prześlij zestawy treningowy, walidacyjny i testowy osobno do magazynu funkcji, używając metody `file_upload` katalogu i podając odpowiednią ścieżkę za każdym razem.
4. Zainicjuj klasyfikator kwantowy, używając akcji `create` funkcji, która przyjmuje hiperparametry, takie jak liczba i typy uczących się modeli, regularyzacja (wartość lambda) oraz opcje optymalizacji, w tym liczba warstw, typ klasycznego optymalizatora, Backend kwantowy itd.
5. Wytrenuj klasyfikator kwantowy na zestawie treningowym, używając akcji `fit` funkcji, podając oznaczony zestaw treningowy i zestaw walidacyjny (jeśli dotyczy).
6. Wykonaj przewidywania na wcześniej niewidzianym zestawie testowym, używając akcji `predict` funkcji.
## Podejście oparte na akcjach
Funkcja korzysta z podejścia opartego na akcjach. Możesz myśleć o niej jak o wirtualnym środowisku, w którym używasz akcji do wykonywania zadań lub zmiany jego stanu. Obecnie oferuje następujące akcje: [list](#1-list), [create](#2-create), [delete](#3-delete), [fit](#4-fit), [predict](#5-predict), [fit_predict](#6-fit-predict) i [create_fit_predict](#7-create-fit-predict). Poniższy przykład demonstruje akcję `create_fit_predict`.

In [ ]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [1, 0, 0, 1, 0]
Probabilities (first five results):  [[0.16849563539001172, 0.8315043646099888], [0.8726393386620336, 0.12736066133796647], [0.795344837290717, 0.20465516270928288], [0.36822585748882725, 0.6317741425111725], [0.6656662698604361, 0.3343337301395641]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 7.945035696029663}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 82.41029238700867}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 77.3459484577179}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 71.27004957199097}}


### 1. List

The `list` action retrieves all stored classifiers in `*.pkl.tar` format from the shared data directory. You can also access the contents of this directory by using the `catalog.files()` method. In general, the list action searches for files with the `*.pkl.tar` extension in the shared data directory and returns them in a list format.

#### Inputs

|     Name    |    Type     | Description |   Required  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | The name of the action from among `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` and `delete`. | Yes |

#### Usage

In [ ]:
job = singularity.run(action="list")

### 1. List

Akcja `list` pobiera wszystkie zapisane klasyfikatory w formacie `*.pkl.tar` ze współdzielonego katalogu danych. Zawartość tego katalogu możesz też przeglądać metodą `catalog.files()`. Ogólnie rzecz biorąc, akcja list wyszukuje pliki z rozszerzeniem `*.pkl.tar` we współdzielonym katalogu danych i zwraca je w postaci listy.

#### Wejście

|     Nazwa   |    Typ      | Opis        |   Wymagane  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | Nazwa akcji spośród: `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` i `delete`. | Tak |

#### Użycie

In [ ]:
job = singularity.run(
    action="create",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
)

### 2. Create
Akcja `create` tworzy klasyfikator określonego typu `quantum_classifier`, korzystając z podanych parametrów, i zapisuje go we współdzielonym katalogu danych.

> **Note:** Funkcja obsługuje obecnie wyłącznie `QuantumEnhancedEnsembleClassifier`.

#### Wejście
|     Nazwa   |    Typ      | Opis        | Wymagane  | Domyślnie |
|-------------|-------------|-------------|-----------|---------|
| `action` | `str` | Nazwa akcji spośród: `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` i `delete`. | Tak | - |
| `name` | `str` | Nazwa klasyfikatora kwantowego, np. `spam_classifier`. | Tak | - |
| `instance` | `str` | Instancja IBM. | Tak | - |
| `backend_name` | `str` | Zasób obliczeniowy IBM. Domyślnie `None`, co oznacza użycie Backend z najmniejszą liczbą oczekujących zadań. | Nie | `None` |
| `quantum_classifier` | `str` | Typ klasyfikatora kwantowego, tj. `QuantumEnhancedEnsembleClassifier`. | Nie | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | Liczba uczących się modeli w zespole. | Nie | `10` |
| `learners_types` | `list` | Typy uczących się modeli. Wśród obsługiwanych typów znajdują się: `DecisionTreeClassifier`, `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier` i `LogisticRegression`. Szczegółowe informacje na temat każdego z nich znajdziesz w [dokumentacji scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Nie | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | Proporcje każdego typu uczącego się modelu w zespole. | Nie | `[1.0]` |
| `learners_options` | `list` | Opcje dla każdego typu uczącego się modelu w zespole. Pełną listę opcji odpowiadających wybranym typom modeli znajdziesz w [dokumentacji scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Nie | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` lub `list` | Typ(y) regularyzacji: `onsite` lub `alpha`. Wartość `onsite` kontroluje człon onsite — wyższe wartości prowadzą do rzadszych zespołów. Wartość `alpha` kontroluje stosunek między członem interakcji a członem onsite — niższe wartości prowadzą do rzadszych zespołów. Jeśli podasz listę, dla każdego typu zostaną wytrenowane modele, a najlepszy zostanie wybrany. | Nie | `onsite` |
| `regularization` | `str` lub `float` lub `list` | Wartość regularyzacji. Ograniczona do przedziału od `0` do `+inf`, jeśli `regularization_type` wynosi `onsite`. Ograniczona do przedziału od `0` do `1`, jeśli `regularization_type` wynosi `alpha`. Jeśli ustawiono `auto`, stosowana jest auto-regularyzacja — optymalny parametr regularyzacji jest wyznaczany metodą przeszukiwania binarnego przy zadanym współczynniku wybranych klasyfikatorów do wszystkich klasyfikatorów (`regularization_desired_ratio`) oraz górnym ograniczeniu parametru regularyzacji (`regularization_upper_bound`). Jeśli podasz listę, dla każdej wartości zostaną wytrenowane modele, a najlepszy zostanie wybrany. | Nie | `0.01` |
| `regularization_desired_ratio` | `float` lub `list` | Żądany współczynnik(i) wybranych klasyfikatorów do wszystkich klasyfikatorów przy auto-regularyzacji. Jeśli podasz listę, dla każdego współczynnika zostaną wytrenowane modele, a najlepszy zostanie wybrany. | Nie | `0.75` |
| `regularization_upper_bound` | `float` lub `list` | Górne ograniczenie(a) parametru regularyzacji przy stosowaniu auto-regularyzacji. Jeśli podasz listę, dla każdego górnego ograniczenia zostaną wytrenowane modele, a najlepszy zostanie wybrany. | Nie | `200` |
| `weight_update_method` | `str` | Metoda aktualizacji wag próbek spośród `logarithmic` i `quadratic`. | Nie | `logarithmic` |
| `sample_scaling` | `boolean` | Czy należy zastosować skalowanie próbek. | Nie | `False` |
| `prediction_scaling` | `float` | Współczynnik skalowania prognoz. | Nie | `None` |
| `optimizer_options` | `dictionary` | Opcje optymalizatora QAOA. Lista dostępnych opcji przedstawiona jest w dalszej części tej dokumentacji. | Nie | ... |
| `voting` | `str` | Użyj głosowania większościowego (`hard`) lub średniej prawdopodobieństw (`soft`) do agregowania prognoz/prawdopodobieństw modeli. | Nie | `hard` |
| `prob_threshold` | `float` | Optymalny próg prawdopodobieństwa. | Nie | `0.5` |
| `random_state` | `integer` | Kontroluj losowość w celu zapewnienia powtarzalności. | Nie | `None` |


- Dodatkowo, opcje `optimizer_options` są wymienione poniżej:

|     Nazwa   |    Typ      | Opis        | Wymagane  | Domyślnie |
|-------------|-------------|-------------|-----------|---------|
| `num_solutions` | `integer` | Liczba rozwiązań | Nie | `1024` |
| `reps` | `integer` | Liczba powtórzeń | Nie | `4` |
| `sparsify` | `float` | Próg rzadkości | Nie | `0.001` |
| `theta` | `float` | Wartość początkowa parametru theta, parametru wariacyjnego QAOA | Nie | `None` |
| `simulator` | `boolean` | Czy używać symulatora czy QPU | Nie | `False` |
| `classical_optimizer` | `str` | Nazwa klasycznego optymalizatora dla QAOA. Można użyć wszystkich solwerów oferowanych przez SciPy, wymienionych [tutaj](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize). Należy odpowiednio ustawić `classical_optimizer_options` | Nie | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | Opcje klasycznego optymalizatora. Pełną listę dostępnych opcji znajdziesz w [dokumentacji SciPy](https://docs.scipy.org/doc/scipy/reference/) | Nie | `{"maxiter": 60}` |
| `optimization_level` | `integer` | Głębokość Circuit QAOA | Nie | `3` |
| `num_transpiler_runs` | `integer` | Liczba uruchomień Transpiler | Nie | `30` |
| `pass_manager_options` | `dictionary` | Opcje generowania menedżera przejść (preset pass manager) | Nie | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | Opcje Estimator. Pełną listę dostępnych opcji znajdziesz w [dokumentacji Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) | Nie | `None` |
| `sampler_options` | `dictionary` | Opcje Sampler. Pełną listę dostępnych opcji znajdziesz w [dokumentacji Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options) | Nie | `None` |

- Domyślne wartości `estimator_options`:

|     Nazwa   |    Typ      | Wartość  |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- Domyślne wartości `sampler_options`:

|     Nazwa   |    Typ      | Wartość |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### Użycie

In [ ]:
job = singularity.run(
    action="delete",
    name="classifier_name",  # specify the name of the classifier to delete here
)

#### Walidacje
- `name`:
    - Nazwa musi być unikalna i zawierać maksymalnie 64 znaki.
    - Może zawierać wyłącznie znaki alfanumeryczne i podkreślenia.
    - Musi zaczynać się od litery i nie może kończyć się podkreśleniem.
    - We współdzielonym katalogu danych nie może już istnieć żaden klasyfikator o tej samej nazwie.

### 3. Delete
Akcja `delete` usuwa klasyfikator ze współdzielonego katalogu danych.

#### Wejście
|     Nazwa   |    Typ      | Opis        |   Wymagane  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Nazwa akcji. Musi wynosić `delete`. | Tak |
| `name`      | `str`    | Nazwa klasyfikatora do usunięcia. | Tak |

#### Użycie

In [ ]:
job = singularity.run(
    action="fit",
    name="classifier_name",  # specify the name of the classifier to train here
    X=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
)

#### Walidacje
- `name`:
    - Nazwa musi być unikalna i zawierać maksymalnie 64 znaki.
    - Może zawierać wyłącznie znaki alfanumeryczne i podkreślenia.
    - Musi zaczynać się od litery i nie może kończyć się podkreśleniem.
    - We współdzielonym katalogu danych musi już istnieć klasyfikator o tej samej nazwie.
### 4. Fit
Akcja `fit` trenuje klasyfikator przy użyciu dostarczonych danych treningowych.

#### Dane wejściowe
|     Nazwa    |    Typ     | Opis |   Wymagane  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Nazwa akcji. Musi mieć wartość `fit`. | Tak |
| `name`      | `str`    | Nazwa klasyfikatora do wytrenowania. | Tak |
| `X`         | `array` lub `list` lub `str` | Dane treningowe. Może to być tablica NumPy, lista lub ciąg znaków wskazujący nazwę pliku w współdzielonym katalogu danych. | Tak |
| `y`         | `array` lub `list` lub `str` | Docelowe wartości treningowe. Może to być tablica NumPy, lista lub ciąg znaków wskazujący nazwę pliku w współdzielonym katalogu danych. | Tak |
| `fit_params`| `dictionary`| Dodatkowe parametry przekazywane do metody `fit` klasyfikatora. | Nie |

##### fit_params
|     Nazwa    |    Typ     | Opis |   Wymagane  |   Domyślnie   |
|-------------|-------------|-------------|-------------|-------------|
| `validation_data` | `tuple` | Dane walidacyjne i etykiety. | Nie | `None` |
| `pos_label` | `integer` lub `str` | Etykieta klasy mapowana na 1. | Nie | `None` |
| `optimization_data` | `str` | Zbiór danych do optymalizacji zespołu. Może przyjmować jedną z wartości: `train`, `validation`, `both`. | Nie | `train` |

#### Użycie

In [ ]:
job = singularity.run(
    action="predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X=X_test,  # or "X_test.npy" if you uploaded it to the shared data directory
    options={
        "out": "output.json",
    },
)

#### Walidacje
- `name`:
    - Nazwa musi być unikalna, ciąg znaków o długości do 64 znaków.
    - Może zawierać wyłącznie znaki alfanumeryczne i podkreślenia.
    - Musi zaczynać się od litery i nie może kończyć się podkreśleniem.
    - Klasyfikator o tej samej nazwie powinien już istnieć we współdzielonym katalogu danych.
### 5. Predict
Akcja `predict` służy do uzyskiwania twardych i miękkich prognoz (prawdopodobieństw).

#### Dane wejściowe
|     Nazwa    |    Typ     | Opis |   Wymagane  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Nazwa akcji. Musi mieć wartość `predict`. | Tak |
| `name`      | `str`    | Nazwa klasyfikatora do użycia. | Tak |
| `X`         | `array` lub `list` lub `str` | Dane testowe. Może to być tablica NumPy, lista lub ciąg znaków wskazujący nazwę pliku w współdzielonym katalogu danych. | Tak |
| `options["out"]` | `str` | Nazwa pliku wyjściowego JSON do zapisania prognoz we współdzielonym katalogu danych. Jeśli nie zostanie podana, prognozy są zwracane w wyniku zadania. | Nie |

#### Użycie

In [ ]:
job = singularity.run(
    action="fit_predict",
    name="classifier_name",  # specify the name of the classifier to use here
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "out": "output.json",
    },
)

#### Walidacje
- `name`:
    - Nazwa musi być unikalna, ciąg znaków o długości do 64 znaków.
    - Może zawierać wyłącznie znaki alfanumeryczne i podkreślenia.
    - Musi zaczynać się od litery i nie może kończyć się podkreśleniem.
    - Klasyfikator o tej samej nazwie powinien już istnieć we współdzielonym katalogu danych.
- `options["out"]`:
    - Nazwa pliku musi być unikalna, ciąg znaków o długości do 64 znaków.
    - Może zawierać wyłącznie znaki alfanumeryczne i podkreślenia.
    - Musi zaczynać się od litery i nie może kończyć się podkreśleniem.
    - Musi mieć rozszerzenie `.json`.
### 6. Fit-predict
Akcja `fit_predict` trenuje klasyfikator przy użyciu danych treningowych, a następnie używa go do uzyskiwania twardych i miękkich prognoz (prawdopodobieństw).

#### Dane wejściowe
|     Nazwa    |    Typ     | Opis |   Wymagane  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Nazwa akcji. Musi mieć wartość `fit_predict`. | Tak |
| `name`      | `str`    | Nazwa klasyfikatora do użycia. | Tak |
| `X_train`   | `array` lub `list` lub `str` | Dane treningowe. Może to być tablica NumPy, lista lub ciąg znaków wskazujący nazwę pliku w współdzielonym katalogu danych. | Tak |
| `y_train`   | `array` lub `list` lub `str` | Docelowe wartości treningowe. Może to być tablica NumPy, lista lub ciąg znaków wskazujący nazwę pliku w współdzielonym katalogu danych. | Tak |
| `X_test`    | `array` lub `list` lub `str` | Dane testowe. Może to być tablica NumPy, lista lub ciąg znaków wskazujący nazwę pliku w współdzielonym katalogu danych. | Tak |
| `fit_params`| `dictionary`| Dodatkowe parametry przekazywane do metody `fit` klasyfikatora. | Nie |
| `options["out"]` | `str` | Nazwa pliku wyjściowego JSON do zapisania prognoz we współdzielonym katalogu danych. Jeśli nie zostanie podana, prognozy są zwracane w wyniku zadania. | Nie |

#### Użycie

In [ ]:
job = singularity.run(
    action="create_fit_predict",
    name="classifier_name",  # specify your custom name for the classifier here
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,  # or "X_train.npy" if you uploaded it in the shared data directory
    y_train=y_train,  # or "y_train.npy" if you uploaded it in the shared data directory
    X_test=X_test,  # or "X_test.npy" if you uploaded it in the shared data directory
    fit_params={},  # define the fit parameters here
    options={
        "save": True,
        "out": "output.json",
    },
)

#### Walidacje
- `name`:
    - Nazwa musi być unikalna, ciąg znaków o długości do 64 znaków.
    - Może zawierać wyłącznie znaki alfanumeryczne i podkreślenia.
    - Musi zaczynać się od litery i nie może kończyć się podkreśleniem.
    - Klasyfikator o tej samej nazwie powinien już istnieć we współdzielonym katalogu danych.

- `options["out"]`:
    - Nazwa pliku musi być unikalna, ciąg znaków o długości do 64 znaków.
    - Może zawierać wyłącznie znaki alfanumeryczne i podkreślenia.
    - Musi zaczynać się od litery i nie może kończyć się podkreśleniem.
    - Musi mieć rozszerzenie `.json`.
### 7. Create-fit-predict
Akcja `create_fit_predict` tworzy klasyfikator, trenuje go przy użyciu dostarczonych danych treningowych, a następnie używa go do uzyskiwania twardych i miękkich prognoz (prawdopodobieństw).

#### Dane wejściowe
| Nazwa | Typ | Opis | Wymagane |
|------|------|-------------|-----------|
| `action` | `str` | Nazwa akcji spośród: `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` i `delete`. | Tak |
| `name` | `str` | Nazwa klasyfikatora do użycia. | Tak |
| `quantum_classifier` | `str` | Typ klasyfikatora, tzn. `QuantumEnhancedEnsembleClassifier`. Domyślnie `QuantumEnhancedEnsembleClassifier`. | Nie |
| `X_train` | `array` lub `list` lub `str` | Dane treningowe. Może to być tablica NumPy, lista lub ciąg znaków wskazujący nazwę pliku w współdzielonym katalogu danych. | Tak |
| `y_train` | `array` lub `list` lub `str` | Docelowe wartości treningowe. Może to być tablica NumPy, lista lub ciąg znaków wskazujący nazwę pliku w współdzielonym katalogu danych. | Tak |
| `X_test` | `array` lub `list` lub `str` | Dane testowe. Może to być tablica NumPy, lista lub ciąg znaków wskazujący nazwę pliku w współdzielonym katalogu danych. | Tak |
| `fit_params` | `dictionary` | Dodatkowe parametry przekazywane do metody `fit` klasyfikatora. | Nie |
| `options["save"]` | `boolean` | Czy zapisać wytrenowany klasyfikator we współdzielonym katalogu danych. Domyślnie `True`. | Nie |
| `options["out"]` | `str` | Nazwa pliku wyjściowego JSON do zapisania prognoz we współdzielonym katalogu danych. Jeśli nie zostanie podana, prognozy są zwracane w wyniku zadania. | Nie |

#### Użycie

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

#### Walidacje
- `name`:
    - Jeśli `options["save"]` ma wartość `True`:
        - Nazwa musi być unikalna, ciąg znaków o długości do 64 znaków.
        - Może zawierać wyłącznie znaki alfanumeryczne i podkreślenia.
        - Musi zaczynać się od litery i nie może kończyć się podkreśleniem.
        - We współdzielonym katalogu danych nie powinien już istnieć klasyfikator o tej samej nazwie.

- `options["out"]`:
    - Nazwa pliku musi być unikalna, ciąg znaków o długości do 64 znaków.
    - Może zawierać wyłącznie znaki alfanumeryczne i podkreślenia.
    - Musi zaczynać się od litery i nie może kończyć się podkreśleniem.
    - Musi mieć rozszerzenie `.json`.
---
## Pierwsze kroki
Uwierzytelnij się przy użyciu [klucza API IBM Quantum Platform](http://quantum.cloud.ibm.com/) i wybierz funkcję Qiskit w następujący sposób:

In [3]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[-0.99958218  0.02890441]
 [ 0.03285169  0.24578719]
 [ 1.13127903 -0.49134546]
 [ 1.86951286  0.00608971]
 [ 0.20190413  0.97940529]
 [ 0.8831311   0.46912627]
 [-0.10819442  0.99412975]
 [-0.20005727  0.97978421]
 [-0.78775705  0.61598607]
 [ 1.82453236 -0.0658148 ]]
Targets: [0 1 1 1 0 0 0 0 0 1]


## Przykład
W tym przykładzie użyjesz funkcji „Singularity Machine Learning - Classification", aby sklasyfikować zbiór danych składający się z dwóch przeplatających się, półkolistych kształtów przypominających księżyce. Zbiór danych jest syntetyczny, dwuwymiarowy i opatrzony binarnymi etykietami. Został stworzony tak, by stanowił wyzwanie dla algorytmów takich jak grupowanie oparte na centroidach czy klasyfikacja liniowa.
![Zbiór danych moons](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
Podczas tego procesu dowiesz się, jak utworzyć klasyfikator, dopasować go do danych treningowych, użyć go do przewidywania na danych testowych oraz usunąć klasyfikator po zakończeniu pracy.
Przed rozpoczęciem musisz zainstalować [scikit-learn](https://scikit-learn.org/). Zainstaluj go przy użyciu następującego polecenia:

In [4]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


Wykonaj następujące kroki:

1) Utwórz syntetyczny zbiór danych przy użyciu funkcji [make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) z biblioteki [scikit-learn](https://scikit-learn.org/).
2) Prześlij wygenerowany syntetyczny zbiór danych do [wspólnego katalogu danych](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html).
3) Utwórz klasyfikator wspomagany kwantowo przy użyciu akcji [create](#2-create).
4) Wypisz swoje klasyfikatory przy użyciu akcji [list](#1-list).
5) Wytrenuj klasyfikator na danych treningowych przy użyciu akcji [fit](#4-fit).
6) Użyj wytrenowanego klasyfikatora do przewidywania na danych testowych przy użyciu akcji [predict](#5-predict).
7) Usuń klasyfikator przy użyciu akcji [delete](#3-delete).
8) Posprzątaj po zakończeniu.

**Krok 1.** Zaimportuj niezbędne moduły i wygeneruj syntetyczny zbiór danych, a następnie podziel go na zbiory treningowy i testowy.

In [5]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [6]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}
['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar', 'my_classifier.pkl.tar']


**Krok 2.** Zapisz oznaczone zbiory treningowy i testowy na lokalnym dysku, a następnie prześlij je do [wspólnego katalogu danych](https://qiskit.github.io/qiskit-serverless/getting_started/experimental/manage_data_directory.html).

In [7]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 8.45469617843628}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 69.4949426651001}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 73.01881957054138}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 75.4787163734436}}}}


**Step 5.** Obtain predictions and probabilities from the quantum-enhanced classifier using the [predict](#5-predict) action.

In [8]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 1, 0, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 0.0], [0.0, 1.0]]


**Krok 3.** Utwórz klasyfikator wspomagany kwantowo przy użyciu akcji [create](#2-create).

In [9]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [ ]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

## Benchmarks

These benchmarks show that the classifier can achieve extremely high accuracies on challenging problems. They also show that increasing the number of learners in the ensemble (number of qubits) can lead to increased accuracy.

"Classical accuracy" refers to the accuracy obtained using corresponding classical state of the art which, in this case, is an AdaBoost classifier based on an ensemble of size 75. "Quantum accuracy", on the other hand, refers to the accuracy obtained using the "Singularity Machine Learning - Classification".

| Problem | Dataset Size | Ensemble Size | Number of Qubits | Classical Accuracy | Quantum Accuracy | Improvement |
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
| Grid stability | 5000 examples, 12 features | 55 | 55 |  76% | 91% | 15% |
| Grid stability | 5000 examples, 12 features | 65 | 65 |  76% | 92% | 16% |
| Grid stability | 5000 examples, 12 features | 75 | 75 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 85 | 85 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 100 | 100 |  76% | 95% | 19% |

----

As quantum hardware evolves and scales, the implications for our quantum classifier become increasingly significant. While the number of qubits does impose limitations on the size of the ensemble that can be utilized, it does not restrict the volume of data that can be processed. This powerful capability enables the classifier to efficiently handle datasets containing millions of data points and thousands of features. Importantly, the constraints related to ensemble size can be addressed through the implementation of a large-scale version of the classifier. By leveraging an iterative outer-loop approach, the ensemble can be dynamically expanded, enhancing flexibility and overall performance. However, it's worth noting that this feature has not yet been implemented in the current version of the classifier.

## Changelog

### 4 June 2025
- Upgraded `QuantumEnhancedEnsembleClassifier` with the following updates:
  - Added onsite/alpha regularization. You can specify `regularization_type` to be `onsite` or `alpha`
  - Added auto-regularization. You can set `regularization` to `auto` to use auto-regularization
  - Added `optimization_data` parameter to the `fit` method to choose optimization data for quantum optimization. You can use one of these options: `train`, `validation`, or `both`
  - Improved overall performance
- Added detailed status tracking for running jobs

### 20 May 2025
- Standardized error handling

### 18 March 2025
- Upgraded qiskit-serverless to 0.20.0 and base image to 0.20.1

### 14 February 2025
- Upgraded base image to 0.19.1

### 6 February 2025
- Upgraded qiskit-serverless to 0.19.0 and base image to 0.19.0

### 13 November 2024
- Release of Singularity Machine Learning - Classification

## Get support

For any questions, [reach out to Multiverse Computing](mailto:singularity@multiversecomputing.com).

Be sure to include the following information:

- The Qiskit Function Job ID (`job.job_id`)
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Multiverse Computing's Singularity Machine Learning Classification function](https://quantum.cloud.ibm.com/functions?id=multiverse-singularity).
- Review [Leclerc, L., et al. (2023). Financial risk management on a neutral atom quantum processor. Physical Review Research, 5, 043117.](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.5.043117)

</Admonition>